# Homework Starter — Stage 05: Data Storage
Name: Jesse Wang
Date: 2026-08-17

Objectives:
- Env-driven paths to `data/raw/` and `data/processed/`
- Save CSV and Parquet; reload and validate
- Abstract IO with utility functions; document choices

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install numpy
# !pip install pandas
# !pip install pyarrow
# !pip install python-dotenv

In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/wangjing/bootcamp_Jesse_Wang/homework/homework5

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [3]:
import os, pathlib, datetime as dt
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
RAW = pathlib.Path(os.getenv('DATA_DIR_RAW', 'data/raw'))
PROC = pathlib.Path(os.getenv('DATA_DIR_PROCESSED', 'data/processed'))
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)
print('RAW ->', RAW.resolve())
print('PROC ->', PROC.resolve())

RAW -> /Users/wangjing/bootcamp_Jesse_Wang/homework/homework5/data/raw
PROC -> /Users/wangjing/bootcamp_Jesse_Wang/homework/homework5/data/processed


## 1) Create or Load a Sample DataFrame
You may reuse data from prior stages or create a small synthetic dataset.

In [4]:
import numpy as np
np.random.seed(5)   # seed so every run produces identical numbers (and files)

dates = pd.date_range('2024-01-01', periods=20, freq='D')
df = pd.DataFrame({'date': dates, 'ticker': ['AAPL']*20, 'price': 150 + np.random.randn(20).cumsum()})
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    20 non-null     datetime64[us]
 1   ticker  20 non-null     str           
 2   price   20 non-null     float64       
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 692.0 bytes


## 2) Save CSV to data/raw/ and Parquet to data/processed/
- Use timestamped filenames.
- Handle missing Parquet engine gracefully.

In [5]:
def ts(): return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

# CSV -> raw (immutable, as-acquired snapshot)
csv_path = RAW / f"sample_{ts()}.csv"
df.to_csv(csv_path, index=False)
print('Saved CSV ->', csv_path)

# Parquet -> processed (columnar, dtype-preserving)
pq_path = PROC / f"sample_{ts()}.parquet"
try:
    df.to_parquet(pq_path)
    print('Saved Parquet ->', pq_path)
except Exception as e:
    print('Parquet engine not available. Install pyarrow or fastparquet to complete this step.')
    print('Error:', e)
    pq_path = None
pq_path

Saved CSV -> data/raw/sample_20260817-175854.csv
Saved Parquet -> data/processed/sample_20260817-175854.parquet


PosixPath('data/processed/sample_20260817-175854.parquet')

## 3) Reload and Validate

In [6]:
def validate_loaded(original, reloaded, cols=('date', 'ticker', 'price')):
    checks = {
        'shape_equal': original.shape == reloaded.shape,
        'cols_present': all(c in reloaded.columns for c in cols),
    }
    if 'date' in reloaded.columns:
        checks['date_is_datetime'] = pd.api.types.is_datetime64_any_dtype(reloaded['date'])
    if 'price' in reloaded.columns:
        checks['price_is_numeric'] = pd.api.types.is_numeric_dtype(reloaded['price'])
    return checks

df_csv = pd.read_csv(csv_path, parse_dates=['date'])
print('CSV validation:')
validate_loaded(df, df_csv)

CSV validation:


{'shape_equal': True,
 'cols_present': True,
 'date_is_datetime': True,
 'price_is_numeric': True}

In [7]:
if pq_path and pq_path.exists():
    try:
        df_pq = pd.read_parquet(pq_path)
        print('Parquet validation:')
        print(validate_loaded(df, df_pq))
        print('\nParquet dtypes (dtype-preserving):')
        print(df_pq.dtypes)
    except Exception as e:
        print('Parquet read failed:', e)
else:
    print('Parquet file not present (skipped earlier).')

Parquet validation:
{'shape_equal': True, 'cols_present': True, 'date_is_datetime': True, 'price_is_numeric': True}

Parquet dtypes (dtype-preserving):
date      datetime64[us]
ticker               str
price            float64
dtype: object


## 4) Utilities
- `detect_format` routes by file suffix; `write_df`/`read_df` use it.
- `write_df` creates missing parent dirs and fails loudly if Parquet has no engine.

In [8]:
import typing as t, pathlib

def detect_format(path: t.Union[str, pathlib.Path]):
    s = str(path).lower()
    if s.endswith('.csv'): return 'csv'
    if s.endswith('.parquet') or s.endswith('.pq') or s.endswith('.parq'): return 'parquet'
    raise ValueError('Unsupported format: ' + s)

def write_df(df: pd.DataFrame, path: t.Union[str, pathlib.Path]):
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    fmt = detect_format(p)
    if fmt == 'csv':
        df.to_csv(p, index=False)
    else:
        try:
            df.to_parquet(p)
        except Exception as e:
            raise RuntimeError('Parquet engine not available. Install pyarrow or fastparquet.') from e
    return p

def read_df(path: t.Union[str, pathlib.Path]):
    p = pathlib.Path(path)
    fmt = detect_format(p)
    if fmt == 'csv':
        out = pd.read_csv(p)
        if 'date' in out.columns:
            out['date'] = pd.to_datetime(out['date'])
        return out
    else:
        try:
            return pd.read_parquet(p)
        except Exception as e:
            raise RuntimeError('Parquet engine not available. Install pyarrow or fastparquet.') from e

# Demo the utilities on both formats
p_csv = RAW / f"util_{ts()}.csv"
p_pq  = PROC / f"util_{ts()}.parquet"

write_df(df, p_csv)
print('Reloaded CSV via util, shape:', read_df(p_csv).shape)

try:
    write_df(df, p_pq)
    print('Reloaded Parquet via util, shape:', read_df(p_pq).shape)
    print('\nread_df(parquet) dtypes:')
    print(read_df(p_pq).dtypes)
except RuntimeError as e:
    print('Skipping Parquet util demo:', e)

Reloaded CSV via util, shape: (20, 3)
Reloaded Parquet via util, shape: (20, 3)

read_df(parquet) dtypes:
date      datetime64[us]
ticker               str
price            float64
dtype: object


## 5) Documentation

Storage choices are documented in **`README.md`** (folder structure, CSV-vs-Parquet
rationale, and how env variables drive the read/write paths). Summary here:

- **Paths** come from `.env` (`DATA_DIR_RAW`, `DATA_DIR_PROCESSED`) with
  `data/raw` / `data/processed` as fallbacks, so the same notebook runs anywhere.
- **CSV → `data/raw/`** (portable, human-readable, diffable) and
  **Parquet → `data/processed/`** (columnar, compressed, dtype-preserving).
- **Validation** reloads every file and checks shape, required columns, and that
  `date` is datetime and `price` is numeric.
- **Utilities** (`write_df`/`read_df`) route on suffix and fail with a clear message
  if Parquet's engine is missing.

Assumptions: RNG is seeded for reproducibility; timestamped filenames avoid
overwrites; Parquet needs `pyarrow`/`fastparquet` installed.